# SPICE confidence sweep

Run All to compare confidence thresholds on the same seeded preliminary samples as the pitch benchmark. The table appears when all tracks finish.

This uses the existing `Spice` adapter's model and conversion. Each worker loads the model once, runs inference once per track, and scores every threshold. Separate raw-output caches make later threshold sweeps cheap. **RPA is unchanged:** rejected pitches are zero Hz, and the usual reference-derived frequency range still applies. No ungated RPA is reported.

The current **0.9** threshold is always included. Results are exploratory sample means; the best RPA and best overall accuracy may choose different thresholds. Production settings and existing benchmark caches are unchanged.

In [1]:
from pathlib import Path
import os
import subprocess
import sys

import pandas as pd
from IPython.display import display

REPO_ROOT = next(
    path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (path / "app.py").is_file() and (path / "benchmarks").is_dir()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from benchmarks.modules.pitch.sweeps.SpiceConfidenceSweep import DEFAULT_THRESHOLDS
from benchmarks.paths import SWEEP_RESULTS_ROOT

DATASETS = ["urmp", "coco", "bach10"]
PER_INSTRUMENT = 2
SEED = 0
WORKERS = min(4, os.cpu_count() or 2)
THRESHOLDS = list(DEFAULT_THRESHOLDS)
OUTPUT_DIR = SWEEP_RESULTS_ROOT / "pitch" / "spice_confidence"
# None uses exactly the existing Spice adapter's TF Hub model/cache.
# A local SavedModel directory can be supplied if the Hub cache is unavailable.
MODEL = None


In [2]:
command = [
    sys.executable, "-m", "benchmarks.modules.pitch.sweeps.SpiceConfidenceSweep",
    "--datasets", *DATASETS,
    "--per-instrument", str(PER_INSTRUMENT),
    "--seed", str(SEED),
    "--workers", str(WORKERS),
    "--thresholds", *(str(t) for t in THRESHOLDS),
    "--output", str(OUTPUT_DIR),
]
if MODEL is not None:
    command += ["--model", str(MODEL)]

# Spawn workers from the importable runner, including on macOS/Jupyter.
# A failed run raises here before any older CSV can be displayed as new results.
subprocess.run(command, cwd=REPO_ROOT, check=True)

rows = pd.read_csv(OUTPUT_DIR / "rows.csv")
summary = pd.read_csv(OUTPUT_DIR / "summary.csv")
columns = [
    "tracks", "Raw Pitch Accuracy", "Overall Accuracy",
    "Voicing Recall", "Voicing False Alarm",
    "rpa_delta_vs_0.9", "oa_delta_vs_0.9",
]
print("Mean scores per dataset and threshold (0–1; deltas versus 0.9):")
display(
    summary.set_index(["dataset", "threshold"])[columns]
    .style.format({c: "{:.4f}" for c in columns if c != "tracks"})
)

winners = []
for dataset, group in summary.groupby("dataset", sort=False):
    for metric in ["Raw Pitch Accuracy", "Overall Accuracy"]:
        best = group.loc[group[metric].idxmax()]
        winners.append({"dataset": dataset, "selected_by": metric,
                        "threshold": best["threshold"],
                        **{c: best[c] for c in columns}})
print("Best thresholds on this sample (ties choose the lowest threshold):")
display(pd.DataFrame(winners).set_index(["dataset", "selected_by"]).round(4))
print(f"Full per-track results: {OUTPUT_DIR / 'rows.csv'}")


60 tracks × 14 thresholds; 0 cached tracks


1/60 tracks scored


2/60 tracks scored


3/60 tracks scored


4/60 tracks scored
5/60 tracks scored


...

         dataset  threshold  Raw Pitch Accuracy  Overall Accuracy  Voicing Recall  Voicing False Alarm  Raw Chroma Accuracy  tracks  rpa_delta_vs_0.9  oa_delta_vs_0.9
bach10-mf0-synth     0.0000              0.9376            0.9135          0.9920               0.3475               0.9403       8            0.1503           0.1224
bach10-mf0-synth     0.1000              0.9375            0.9168          0.9907               0.3047               0.9401       8            0.1502           0.1256
bach10-mf0-synth     0.2000              0.9371            0.9177          0.9819               0.2884               0.9393       8            0.1498           0.1266
bach10-mf0-synth     0.3000              0.9370            0.9194          0.9721               0.2672               0.9387       8            0.1497           0.1282
bach10-mf0-synth     0.4000              0.9359            0.9189          0.9595               0.2602               0.9376       8            0.1486           0.127

Mean scores per dataset and threshold (0–1; deltas versus 0.9):


Best thresholds on this sample (ties choose the lowest threshold):


threshold  tracks  Raw Pitch Accuracy  \
dataset          selected_by                                                 
bach10-mf0-synth Raw Pitch Accuracy        0.0       8              0.9376   
                 Overall Accuracy          0.3       8              0.9370   
coco             Raw Pitch Accuracy        0.0      26              0.9381   
                 Overall Accuracy          0.3      26              0.9368   
urmp             Raw Pitch Accuracy        0.1      26              0.8623   
                 Overall Accuracy          0.4      26              0.8570   

                                     Overall Accuracy  Voicing Recall  \
dataset          selected_by                                            
bach10-mf0-synth Raw Pitch Accuracy            0.9135          0.9920   
                 Overall Accuracy              0.9194          0.9721   
coco             Raw Pitch Accuracy            0.9183          0.9884   
                 Overall Accuracy              0.9218          0.9763   
urmp             Raw Pitch Accuracy            0.8162          0.9613   
                 Overall Accuracy              0.8282          0.9163   

                                     Voicing False Alarm  rpa_delta_vs_0.9  \
dataset          selected_by                                                 
bach10-mf0-synth Raw Pitch Accuracy               0.3475            0.1503   
                 Overall Accuracy                 0.2672            0.1497   
coco             Raw Pitch Accuracy               0.4566            0.1756   
                 Overall Accuracy                 0.3646            0.1742   
urmp             Raw Pitch Accuracy               0.3410            0.1827   
                 Overall Accuracy                 0.2670            0.1774   

                                     oa_delta_vs_0.9  
dataset          selected_by                          
bach10-mf0-synth Raw Pitch Accuracy           0.1224  
                 Overall Accuracy             0.1282  
coco             Raw Pitch Accuracy           0.1507  
                 Overall Accuracy             0.1542  
urmp             Raw Pitch Accuracy           0.0958  
                 Overall Accuracy             0.1079

Full per-track results: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/sweeps/pitch/spice_confidence/rows.csv
